<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/LLM_Trend_%E1%84%91%E1%85%B3%E1%84%85%E1%85%A9%E1%84%8C%E1%85%A6%E1%86%A8%E1%84%90%E1%85%B3_A_A1_B_C_%E1%84%87%E1%85%A9%E1%86%A8%E1%84%89%E1%85%A1%E1%84%87%E1%85%A9%E1%86%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KoChatGPT 프로젝트: A · A-1 · B · C 비교

첨부 예제의 **SFT → RM(보상 모델) → PPO** 흐름을 재사용하는 프로젝트 답안이다.

| 실험 | 모델 | SFT | RM | PPO 질문 |
|---|---|---|---|---|
| A | KoGPT-2 | 기존 데이터, 1 epoch | 기존 순위 데이터, 1 epoch | 기존 질문 |
| A-1 | EXAONE-4.0-1.2B | 추가 학습 없이 평가 | — | — |
| B | EXAONE | 정제 기존 데이터 + KoAlpaca 최대 5,000개, 1 epoch | 기존 데이터 + 한국어 UltraFeedback 최대 2,000쌍, 1 epoch | 기존 질문 + 선택한 KoAlpaca 질문 재사용 |
| C | EXAONE | B와 같은 데이터, 3 epochs | B와 같은 학습된 RM 재사용 | B와 동일 |

질문 중복을 정리한 뒤 **전체 데이터에 하나의 train/validation/test 질문 분할**을 적용한다.
같은 질문이 여러 데이터셋·학습 단계·RM 답변 쌍에 등장해도 같은 분할에 속한다.
KoAlpaca 학습 5,000개, 공통 검증 200문항, 최종 평가 50문항은 따로 확보한다.
정제·중복 제거·길이 제한으로 최종 사용량은 줄어들 수 있으며 실제 개수를 출력한다.

기본 `RUN_ALL=True`에서 A → A-1 → B → C 학습·검증을 자동으로 실행한다.
이어서 B/C의 **SFT 검증 ROUGE-L**로 1·3 epochs 중 하나를 선택하고(동점이면 1 epoch), 네 비교군의 동일한 최종 50문항 평가까지 진행한다.
완료된 모델과 평가 파일은 재사용한다. 한 실험씩 직접 실행하려면 `RUN_ALL=False`로 두고 `PHASE`, `EXPERIMENT`를 선택한다.

A-1은 배포된 EXAONE 4.0 1.2B 모델의 추가 학습 전 성능이다. A-1·B·C는 모델 버전·BF16(16비트) 로딩·대화 템플릿을 맞춘다.
일반 질문·답변 비교를 위해 모든 단계에 `enable_thinking=False`를 적용한다.
C는 원래 EXAONE에서 시작한다. B의 PPO 모델을 이어 학습하지 않는다.
같은 실험에서는 `RUN_NAME`과 작업 폴더를 유지한다. 기존 답안 결과와 섞이지 않도록 새 기본 RUN_NAME을 사용한다.
모델 가중치 다운로드와 GPU 학습은 사용자가 아래 셀을 실행할 때 시작한다.


## 예제에서 가져온 코드와 필요한 변경

`Based-On: none` — 로컬 `EXAMPLES_INDEX.md`와 `catalog.json`에 이 RLHF 예제가 없어, 첨부 노트북을 직접 사용했다.
출처 종류는 `project_extension`이다. 첨부 파일은 사용자 상태를 포함할 수 있는 LMS 내보내기 자료다.

| 재사용 | 첨부 노트북의 코드 셀 위치(0부터) | 변경 |
|---|---|---|
| `SFT_dataset`, `DataCollatorForSupervisedDataset` | 50, 51 | EXAONE 대화 템플릿 연결, 나머지 토큰화·정답 마스킹·패딩 유지 |
| `TrainingArguments`, `Trainer` | 56, 59 | 실험별 경로·epoch, EXAONE LoRA의 실제 batch 1 × 누적 8 |
| 3개 답변 → chosen/rejected 3쌍 | 75, 79, 80 | 질문 그룹 분할 후 쌍 생성, UltraFeedback 추가 |
| `RewardDataset`, `RewardModelTrainer` | 67, 84, 85 | 같은 Trainer 사용, EOS 토큰과 점수층 저장 보완 |
| `Actor`, `Critic`, `RewardModel`, `PPOTrainer` | 95–111 | GPT2 전용 로더를 공통 로더로 연결, 최신 generate 연결 |

학습 알고리즘과 loss(손실 함수)는 예제 저장소의 구현을 그대로 사용한다. TRL로 새 학습 루프를 작성하지 않는다.
필수 보완은 A·B·C에 공통 적용한다: 한국어 Fast tokenizer(고속 토크나이저), 단계 간 prompt 통일,
RM 점수층 저장·복원, padding을 제외한 보상 평균, 생성 시 왼쪽 패딩에 맞춘 position ID(위치 ID).
따라서 **A는 예제 설정에 이 보완을 적용한 재학습 기준선**이다. 이 프로젝트의 같은 RUN_NAME 아래 저장된 체크포인트를 재사용한다. 다른 설정·경로의 체크포인트는 별도로 취급한다.

EXAONE의 네 모델(actor·critic·고정 RM·고정 reference)은 각각 별도로 로드한다.
학습 대상과 고정 모델의 파라미터는 각 인스턴스에 별도로 둔다.
EXAONE 기반 가중치는 BF16으로 고정하며, SFT/actor는 LoRA만 학습한다.
RM/critic은 LoRA와 FP32 점수층을 학습하고, 고정 RM/reference는 모든 가중치를 고정한다.
작은 LoRA 가중치는 PEFT 기본값인 FP32로 유지한다. PPO 로그 확률도 FP32로 계산한다.
검증 범위는 구조·문법, 자동 실행 순서·단계 재사용·실패/중단 시 재실행, 작은 모델의 마스킹·보상 저장/복원·소규모 PPO 갱신이다. 실제 GPU 메모리와 전체 학습은 별도 확인한다.


## 1. 설치와 실험 선택

Colab의 **A100 40GB GPU** 런타임에서 실행한다. EXAONE은 **BF16 + LoRA**를 사용한다.
4비트 양자화 설정과 bitsandbytes 설치를 제거했다. 학습 시 gradient checkpointing(중간값 재계산)을 사용한다.
설치 셀은 프로젝트에서 사용하지 않는 `diffusers`, `gradio`, `gcsfs`를 현재 런타임에서 제거하고 필요한 패키지를 한 번에 설치한다.
설치 마지막의 `pip check` 결과를 확인한 뒤 **런타임을 한 번 재시작하고 다음 셀부터 실행**한다.
`datasets==3.6.0`에 맞춰 `fsspec==2025.3.0`을 고정한다. Drive는 `google.colab.drive`로 연결한다.
다음 설정 셀에서 Google Drive를 연결한다. 표시되는 Google 계정 접근 요청을 허용한다.
데이터는 `내 드라이브/llm_trend_project/<RUN_NAME>/data`에 저장하며, 같은 `RUN_NAME`으로 다시 실행하면 재사용한다.
모델과 결과는 `llm_trend_project/<RUN_NAME>/a100_exaone4_1_2b_lora/` 아래 `models`, `results`에 저장한다.
같은 폴더에 완료된 모델·평가 파일이 있으면 재사용하고, 남은 A·A-1·B·C 단계를 순서대로 실행한다.
EXAONE 4.0 지원에 필요한 Transformers 4.54.0 이상 중 **4.54.1**로 고정했다.
연결되는 `huggingface-hub`도 0.34.4로 맞췄다. 모델과 토크나이저는 같은 고정 버전을 읽는다. 기존 설치가 다른 실습과 충돌하면 새 런타임에서 진행한다.


In [ ]:
# 이 프로젝트 전용 Colab 런타임: 사용하지 않는 충돌 패키지를 제거한다.
!pip uninstall -q -y diffusers gradio gcsfs

# 전체 버전 조건을 한 번에 맞춘다.
!pip install -q loralib==0.1.2 transformers==4.54.1 \
    peft==0.15.2 accelerate==1.6.0 huggingface-hub==0.34.4 \
    datasets==3.6.0 fsspec==2025.3.0 rouge-score==0.1.2

!pip check


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following de

In [23]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")  # 처음 연결할 때 Google 계정 접근을 허용한다.

RUN_ALL = True                    # True: 전체 자동 실행 / False: EXPERIMENT·PHASE로 수동 실행
EXPERIMENT = "A"                    # "A", "A-1", "B", "C"
PHASE = "train"                     # "train": 학습·검증 / "final": 선택 완료 후 최종 평가
RUN_NAME = "comparison_02_group_split"          # 새 실험에서는 이름을 바꿔 기존 결과 보존
# BF16 + LoRA 모델·결과를 이전 실행과 구분한다. Drive 데이터 경로는 RUN_NAME으로 공유한다.
ROOT = Path("llm_trend_project") / RUN_NAME / "a100_exaone4_1_2b_lora"
ROOT.mkdir(parents=True, exist_ok=True)  # 예제 코드는 Colab 로컬에서 실행한다.
ARTIFACT_ROOT = Path("/content/drive/MyDrive/llm_trend_project") / RUN_NAME / ROOT.name
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
SFT_SAVE_STEPS = 200                # optimizer update 기준, 최근 2개 보관
SEED = 230319                      # 예제 RM shuffle seed
MODEL_IDS = {"kogpt2": "skt/kogpt2-base-v2",
             "exaone": "LGAI-EXAONE/EXAONE-4.0-1.2B"}
EXAONE_REVISION = "3abf2810673c7c0778df64a73c2d52eab32d91c4"
KOGPT_REVISION = "d0c0df48bf2b2c9350dd855021a5b216f560c0c7"
KOALPACA_REVISION = "03d3122c530b1e47195c08a3d851eeadddad9689"
ULTRAFEEDBACK_ID = "maywell/ko_Ultrafeedback_binarized"
ULTRAFEEDBACK_REVISION = "43104873ebd2fce703c797a037b97f0b351c2ed3"

# 데이터 수량: 학습용 추가분과 검증·최종 평가를 각각 확보한다.
KOALPACA_TRAIN_SIZE = 5000
ULTRAFEEDBACK_TRAIN_PAIRS = 2000
VALIDATION_FRACTION = 0.10           # 전체 질문 그룹 중 검증 분할 비율
VALIDATION_SIZE = 200               # 답변 생성으로 비교할 공통 검증 문항
FINAL_TEST_SIZE = 50
RM_ORIGINAL_TRAIN_PAIRS = 1000
RM_VALIDATION_PAIRS = 200           # 기존 RM 200쌍 + B/C는 UltraFeedback 200쌍

# 예제 기반 시작값. 하이퍼파라미터를 바꾸면 새 RUN_NAME으로 비교한다.
SFT_LR, RM_LR = 5e-5, 5e-5
PPO_ACTOR_LR, PPO_CRITIC_LR = 5e-6, 5e-6
MAX_SEQ_LEN, PPO_PROMPT_LEN, PPO_TOTAL_LEN = 512, 96, 128
EVAL_MAX_NEW_TOKENS = 128
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
PPO_BATCH_SIZE, PPO_EXPERIENCE_BATCH_SIZE = 8, 8
PPO_EPOCHS, PPO_EPISODES, PPO_TIMESTEPS, PPO_UPDATE_TIMESTEPS = 1, 10, 3, 3
PPO_KL_COEF, PPO_EPS_CLIP, PPO_VALUE_CLIP = 0.1, 0.2, 0.4

KOCHAT_COMMIT = "5d01e3d74d5ef5a0a32c18150dc9b907eef3f691"

EXPERIMENTS = {
    "A":   {"family": "kogpt2", "sft_epochs": 1, "data": "sft_original"},
    "A-1": {"family": "exaone", "sft_epochs": 0, "data": None},
    "B":   {"family": "exaone", "sft_epochs": 1, "data": "sft_improved"},
    "C":   {"family": "exaone", "sft_epochs": 3, "data": "sft_improved"},
}
def select_experiment(name):
    global EXPERIMENT, CFG, IS_EXAONE, DO_TRAIN, MODEL_ID, MODEL_DIR, SFT_DIR, PPO_DIR, RM_DIR
    EXPERIMENT = name
    CFG = EXPERIMENTS[EXPERIMENT]
    if PHASE not in {"train", "final"}:
        raise ValueError('PHASE는 "train" 또는 "final"입니다.')
    IS_EXAONE = CFG["family"] == "exaone"
    DO_TRAIN = EXPERIMENT != "A-1"
    MODEL_ID = MODEL_IDS[CFG["family"]]
    MODEL_DIR = ARTIFACT_ROOT / "models" / EXPERIMENT
    SFT_DIR, PPO_DIR = MODEL_DIR / "sft", MODEL_DIR / "ppo"
    RM_DIR = ARTIFACT_ROOT / "models" / ("rm_exaone_shared" if IS_EXAONE else "rm_kogpt2")

select_experiment(EXPERIMENT)
DATA_DIR = Path("/content/drive/MyDrive/llm_trend_project") / RUN_NAME / "data"
RESULT_DIR = ARTIFACT_ROOT / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

print("Google Drive 데이터 저장 위치:", DATA_DIR)
print("Google Drive 모델·체크포인트·결과:", ARTIFACT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 데이터 저장 위치: /content/drive/MyDrive/llm_trend_project/comparison_02_group_split/data
Google Drive 모델·체크포인트·결과: /content/drive/MyDrive/llm_trend_project/comparison_02_group_split/a100_exaone4_1_2b_lora


In [ ]:
# 예제의 git clone + chatgpt 복사. 전용 폴더만 사용한다.
import subprocess
import importlib
import shutil
import sys

REPO = ROOT / "KoChatGPT"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/airobotlab/KoChatGPT", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", KOCHAT_COMMIT], check=True)
revision = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if revision != KOCHAT_COMMIT:
    raise ValueError("KoChatGPT 버전이 다릅니다. 새 RUN_NAME으로 진행하세요.")
VENDOR = ROOT / "vendor"
CHATGPT = VENDOR / "chatgpt"
if not CHATGPT.exists():
    shutil.copytree(REPO / "colossalai_ChatGPT_230319/chatgpt", CHATGPT)
sys.path.insert(0, str(VENDOR.resolve()))

# 첨부 예제의 ColossalAI import 패치를 줄 번호 대신 문자열로 적용한다.
patches = {
    "trainer/strategies/__init__.py": [
        ("from .colossalai import ColossalAIStrategy\n", ""),
        (", 'ColossalAIStrategy'", ""),
    ],
    "trainer/callbacks/save_checkpoint.py": [
        ("import ColossalAIStrategy, Strategy", "import Strategy"),
        ("not isinstance(self.strategy, ColossalAIStrategy)", "True"),
    ],
    "replay_buffer/utils.py": [
        ("set(('action_log_probs', 'action_mask'))", "set(('sequences', 'action_log_probs', 'attention_mask', 'action_mask'))"),
    ],
    "dataset/reward_dataset.py": [
        ('+ "<|endoftext|>"', '+ tokenizer.eos_token'),
    ],
}
for relative, changes in patches.items():
    path = CHATGPT / relative
    source = path.read_text()
    for old, new in changes:
        source = source.replace(old, new)
    path.write_text(source)

# 이미 import한 커널에서도 버퍼의 배치 구성 함수가 갱신되도록 한다.
for module_name in ["chatgpt.replay_buffer.utils", "chatgpt.replay_buffer.naive"]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])


In [ ]:
import copy
import gc
import hashlib
import json
import logging
import random
import traceback
import re
import unicodedata
from dataclasses import dataclass
from typing import Dict, Sequence

import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast
from transformers import set_seed
from peft import LoraConfig, PeftModel, get_peft_model
from datasets import load_dataset
from rouge_score import rouge_scorer
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import Actor, Critic, RewardModel
from chatgpt.models.utils import log_probs_from_logits
from chatgpt.trainer import RewardModelTrainer, PPOTrainer
from chatgpt.trainer.strategies import NaiveStrategy

print("torch:", torch.__version__, "transformers:", transformers.__version__)
if transformers.__version__ != "4.54.1":
    raise RuntimeError("설치 셀 실행 후 런타임을 재시작하세요.")
if not torch.cuda.is_available():
    raise RuntimeError("실제 학습에는 NVIDIA GPU 런타임을 선택하세요.")
if (RUN_ALL or IS_EXAONE) and not torch.cuda.is_bf16_supported():
    raise RuntimeError("EXAONE BF16 학습에는 A100 GPU 런타임을 선택하세요.")
print("GPU:", torch.cuda.get_device_name(0))
device = "cuda"
set_seed(SEED)


torch: 2.11.0+cu128 transformers: 4.54.1
GPU: NVIDIA A100-SXM4-40GB


## 2. 데이터 준비: 질문 그룹으로 먼저 분리

KoAlpaca는 `instruction/output`, 한국어 UltraFeedback은 `prompt/chosen/rejected` 문자열을 사용한다.
각 출처의 고정 버전과 실제 schema(필드 구조)에 맞춰 읽는다.
처음 실행하면 예제 원자료 3개, KoAlpaca·UltraFeedback JSON과 datasets 캐시를 Drive에 저장한다.
분할 전에 추가 데이터 전체를 받아 질문 중복을 확인하고, 그중 학습용 5,000개·2,000쌍을 선택한다.
아래에서 만드는 학습·검증·최종 평가 JSON도 같은 Drive 데이터 폴더에 저장한다.

1. 빈 질문·빈 답변을 제거하고 SFT/PPO는 질문 기준, UltraFeedback은 질문·답변 쌍 기준으로 중복을 제거한다.
2. KoAlpaca에서 최종 평가 50문항을 먼저 확보한다.
3. 남은 **전체 출처의 질문 키** 중 10%를 검증 그룹으로 지정하고, 나머지를 학습 그룹으로 지정한다.
4. 학습 그룹에서 KoAlpaca 최대 5,000개와 UltraFeedback 최대 2,000쌍을 선택한다.
5. 검증 그룹의 KoAlpaca에서 공통 검증 200문항을 선택한다. RM도 검증 그룹의 쌍만 사용한다.

질문 정규화는 Unicode·공백·문장부호 차이를 처리한다. 뜻만 같은 paraphrase(바꿔 쓴 질문)는 별도 검토 대상이다.
배포 모델의 사전학습 데이터와의 중복 여부는 확인할 수 없다.
도입문 정제는 답변 앞의 인공지능 자기소개에만 적용하고, 답변 중간 설명과 정당한 거절은 유지한다.
참고 답변의 사실 정확성은 모델 결과를 보기 전에 확인한다. 분할·참고 답변은 이후 고정한다.


In [ ]:
def normalize_question(text):
    return re.sub(r"[\W_]+", "", unicodedata.normalize("NFKC", text).casefold())

INTRO = re.compile(
    r"^\s*['\"‘’“”]?(?:저는|나는|제가)\s*(?:AI|인공지능)(?:\s*언어)?"
    r"(?:\s*(?:모델|챗봇|어시스턴트))?[^.!?\n]*[.!?]\s*", re.IGNORECASE)

def clean_answer(text):
    cleaned = INTRO.sub("", text, count=1).strip()
    return cleaned or text.strip()     # 답변 전체가 사라지면 원문 보존

def read_json(path):
    with open(path, encoding="utf-8-sig") as file:
        return json.load(file)         # 예제의 .jsonl은 실제로 JSON 배열

def save_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def deduplicate(rows, pair=False):
    unique = {}
    for row in rows:
        key = normalize_question(row["prompt"])
        fields = ["chosen", "rejected"] if pair else (["completion"] if "completion" in row else [])
        if not key or any(not row[field].strip() for field in fields):
            continue
        if pair and row["chosen"].strip() == row["rejected"].strip():
            continue
        identity = (key, row["chosen"].strip(), row["rejected"].strip()) if pair else key
        unique.setdefault(identity, row)
    return list(unique.values())

def take_rows(rows, limit):
    rows = list(rows)
    random.Random(SEED).shuffle(rows)
    return rows[:limit]


In [ ]:
# 예제 저장소에서 받은 원자료도 Drive에 보관하고, 이후에는 저장된 파일을 읽는다.
kochat_dir = DATA_DIR / "kochatgpt"
kochat_dir.mkdir(exist_ok=True)
for name in ["kochatgpt_1_SFT.jsonl", "kochatgpt_2_RM.jsonl", "kochatgpt_3_PPO.jsonl"]:
    if not (kochat_dir / name).exists():
        shutil.copy2(REPO / "data_kochatgpt" / name, kochat_dir / name)
raw_sft = deduplicate(read_json(kochat_dir / "kochatgpt_1_SFT.jsonl"))
raw_rm = read_json(kochat_dir / "kochatgpt_2_RM.jsonl")
raw_ppo = deduplicate(read_json(kochat_dir / "kochatgpt_3_PPO.jsonl"))

# 추가 데이터와 datasets 캐시를 Drive에 저장한다. JSON이 있으면 다운로드를 건너뛴다.
alpaca_path, uf_path = DATA_DIR / "koalpaca.json", DATA_DIR / "ultrafeedback.json"
if not alpaca_path.exists():
    ds = load_dataset("beomi/KoAlpaca-v1.1a", revision=KOALPACA_REVISION, split="train",
                      cache_dir=str(DATA_DIR / "hf_datasets_cache"))
    save_json(alpaca_path, [{"prompt": row["instruction"], "completion": row["output"]} for row in ds])
if not uf_path.exists():
    ds = load_dataset(ULTRAFEEDBACK_ID, revision=ULTRAFEEDBACK_REVISION, split="train",
                      cache_dir=str(DATA_DIR / "hf_datasets_cache"))
    save_json(uf_path, [{key: row[key] for key in ["prompt", "chosen", "rejected"]} for row in ds])
alpaca_rows = deduplicate(read_json(alpaca_path))
uf_rows = deduplicate(read_json(uf_path), pair=True)

split_config = dict(seed=SEED, validation_fraction=VALIDATION_FRACTION,
    validation_size=VALIDATION_SIZE, final_test_size=FINAL_TEST_SIZE,
    koalpaca_train_size=KOALPACA_TRAIN_SIZE, ultrafeedback_train_pairs=ULTRAFEEDBACK_TRAIN_PAIRS,
    rm_original_train_pairs=RM_ORIGINAL_TRAIN_PAIRS, rm_validation_pairs=RM_VALIDATION_PAIRS,
    koalpaca_revision=KOALPACA_REVISION, ultrafeedback_revision=ULTRAFEEDBACK_REVISION,
    kochat_commit=KOCHAT_COMMIT)
split_path = DATA_DIR / "question_splits.json"
if not split_path.exists():
    final_rows = take_rows(alpaca_rows, FINAL_TEST_SIZE)
    test_keys = {normalize_question(row["prompt"]) for row in final_rows}
    all_keys = {normalize_question(row["prompt"]) for rows in
                [raw_sft, raw_rm, raw_ppo, alpaca_rows, uf_rows] for row in rows}
    candidates = sorted(all_keys - test_keys - {""})
    random.Random(SEED).shuffle(candidates)
    val_keys = set(candidates[:int(len(candidates) * VALIDATION_FRACTION)])
    split_map = {key: "test" if key in test_keys else "validation" if key in val_keys else "train"
                 for key in all_keys if key}
    val_rows = take_rows([row for row in alpaca_rows
                         if split_map[normalize_question(row["prompt"])] == "validation"], VALIDATION_SIZE)
    if len(final_rows) != FINAL_TEST_SIZE or len(val_rows) != VALIDATION_SIZE:
        raise ValueError("최종 평가 또는 검증 문항이 부족합니다. 분할 비율과 원자료를 확인하세요.")
    for name, rows in [("evaluation_50", final_rows), ("validation_questions", val_rows)]:
        save_json(DATA_DIR / f"{name}.json", [
            {"prompt": row["prompt"], "reference": clean_answer(row["completion"])} for row in rows])
    save_json(split_path, {"config": split_config, "question_split": split_map})
split_info = read_json(split_path)
if split_info["config"] != split_config:
    raise ValueError("저장된 분할 설정과 다릅니다. 새 RUN_NAME을 사용하세요.")
split_map = split_info["question_split"]

def in_split(rows, name):
    return [row for row in rows if split_map.get(normalize_question(row["prompt"])) == name]

validation_questions = read_json(DATA_DIR / "validation_questions.json")
evaluation = read_json(DATA_DIR / "evaluation_50.json")
for name, rows, count in [("validation", validation_questions, VALIDATION_SIZE),
                          ("test", evaluation, FINAL_TEST_SIZE)]:
    if len(rows) != count or len({normalize_question(row["prompt"]) for row in rows}) != count:
        raise ValueError(f"{name}: 문항 수 또는 중복 질문을 확인하세요.")
    if len(in_split(rows, name)) != count or any(not row["reference"].strip() for row in rows):
        raise ValueError(f"{name}: 저장된 질문 분할 또는 참고 답변이 잘못되었습니다.")

sft_original = in_split(raw_sft, "train")
original_keys = {normalize_question(row["prompt"]) for row in sft_original}
alpaca_train = take_rows([row for row in in_split(alpaca_rows, "train")
                         if normalize_question(row["prompt"]) not in original_keys], KOALPACA_TRAIN_SIZE)
uf_train = take_rows(in_split(uf_rows, "train"), ULTRAFEEDBACK_TRAIN_PAIRS)
uf_validation = take_rows(in_split(uf_rows, "validation"), RM_VALIDATION_PAIRS)
sft_cleaned = [{"prompt": row["prompt"], "completion": clean_answer(row["completion"])}
               for row in sft_original]
sft_improved = sft_cleaned + alpaca_train
ppo_original = in_split(raw_ppo, "train")
ppo_improved = deduplicate(ppo_original + [{"prompt": row["prompt"]} for row in alpaca_train])
ppo_rows = ppo_improved if IS_EXAONE else ppo_original
rm_rows = in_split(raw_rm, "train") + in_split(raw_rm, "validation")
save_json(DATA_DIR / "sft_original.json", sft_original)
save_json(DATA_DIR / "sft_improved.json", sft_improved)
save_json(DATA_DIR / "koalpaca_train.json", alpaca_train)
save_json(DATA_DIR / "ultrafeedback_train.json", uf_train)

datasets_used = {"A SFT": sft_original, "B/C SFT": sft_improved,
                "RM 기존 학습 후보": in_split(raw_rm, "train"), "RM UltraFeedback 추가": uf_train,
                "A PPO": ppo_original, "B/C PPO": ppo_improved}
data_summary = []
for name, rows in datasets_used.items():
    invalid = len(rows) - len(in_split(rows, "train"))
    if invalid:
        raise ValueError(f"{name}: 검증/최종 평가 질문이 {invalid}개 섞였습니다.")
    data_summary.append({"데이터": name, "개수": len(rows), "검증·최종 평가 중복": invalid})
display(pd.DataFrame(data_summary))
save_json(DATA_DIR / "data_summary.json", data_summary)
print("별도 확보:", len(validation_questions), "검증 문항 /", len(evaluation), "최종 평가 문항")
print("추가 학습 데이터:", len(alpaca_train), "KoAlpaca /", len(uf_train), "UltraFeedback 쌍")
display(pd.DataFrame(validation_questions).head())
DATA_SIGNATURE = hashlib.sha256(json.dumps(
    [split_config, validation_questions, evaluation], ensure_ascii=False, sort_keys=True).encode()).hexdigest()


,데이터,개수,검증·최종 평가 중복
0,A SFT,10702,0
1,B/C SFT,15702,0
2,RM 기존 학습 후보,9155,0
3,RM UltraFeedback 추가,2000,0
4,A PPO,10702,0
5,B/C PPO,15702,0


별도 확보: 200 검증 문항 / 50 최종 평가 문항
추가 학습 데이터: 5000 KoAlpaca / 2000 UltraFeedback 쌍


,prompt,reference
0,"'트램'의 기원, 종류, 장단점 등에 대한 자료를 어디서 구할 수 있을까요? 관련 ...",노면전차(tram)는 19세기 말 도로교통 근대화의 한 방편으로 미국에서 처음으로 ...
1,"최초의 계단은 언제, 어떻게 만들어졌나요?",계단은 인류가 높이와 경사가 있는 건축물을 만든 시점부터 존재하였습니다. BC 30...
2,벌이 꿀을 어떻게 옮기는 걸까요? 꿀은 어떤 과정을 거쳐서 만들어지나요? 꿀을 먹어...,벌이 꿀을 만들어내는 과정은 꽃꿀을 모아 숙성시켜서 축적하는 과정입니다. 꽃꿀은 꿀...
3,화분에 물을 주면 물이 흐르도록 하는 이유는 무엇인가요?,"화분 속에 신선한 산소를 공급하고, 식물이 내놓은 노폐물을 배출하기 위해서 물이 물..."
4,비상착륙 시 사막에서는 바퀴를 넣고 착륙하는 건가요?,"네, 사막에서 비상착륙 시에도 항공기는 바퀴를 넣고 착륙합니다. 바퀴를 넣는 이유는..."


## 3. 모델·토크나이저 연결

KoGPT-2는 예제의 token ID 체계를 읽는 `PreTrainedTokenizerFast`를 사용한다.
EXAONE 4.0에는 공식 대화 템플릿의 일반 응답 모드(`enable_thinking=False`)를 SFT·RM·PPO·평가에 동일하게 적용한다.
공식 예제처럼 user(사용자) 질문으로 시작한다. 템플릿의 닫힌 `<think>` 블록은 prompt에 포함되며, 평가 시에는 새로 생성한 답변만 사용한다.
SFT/RM 최대 길이 512, PPO 입력 96·전체 128 토큰은 예제를 따른다.


In [ ]:
def prepare_tokenizer():
    if IS_EXAONE:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=EXAONE_REVISION,
                                                  model_max_length=MAX_SEQ_LEN, padding_side="right")
    else:
        tokenizer = PreTrainedTokenizerFast.from_pretrained(
            MODEL_ID, revision=KOGPT_REVISION, bos_token="</s>", eos_token="</s>",
            unk_token="<unk>", pad_token="<pad>", mask_token="<mask>",
            model_max_length=MAX_SEQ_LEN, padding_side="right")

    tokenizer.model_input_names = ["input_ids", "attention_mask"]
    sample_text = "안녕하세요. 한국어 문장을 확인합니다."
    decoded = tokenizer.decode(tokenizer.encode(sample_text, add_special_tokens=False), skip_special_tokens=True)
    print("한국어 복원:", decoded)
    if decoded != sample_text or tokenizer.pad_token_id == tokenizer.eos_token_id:
        raise ValueError("한국어 복원 또는 PAD/EOS 설정을 확인하세요.")

    return tokenizer

def make_prompt(question):
    if IS_EXAONE:
        return tokenizer.apply_chat_template([
            {"role": "user", "content": question},
        ], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return "### Instruction(명령어):\n" + question + "\n\n### Response(응답):"

if not RUN_ALL:
    tokenizer = prepare_tokenizer()


In [ ]:
def load_model(checkpoint=None, backbone=False, trainable=True):
    if not IS_EXAONE:
        factory = AutoModel if backbone else AutoModelForCausalLM
        kwargs = {} if checkpoint else {"revision": KOGPT_REVISION}
        model = factory.from_pretrained(str(checkpoint) if checkpoint else MODEL_ID, **kwargs).to(device)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=EXAONE_REVISION,
            torch_dtype=torch.bfloat16, device_map={"": 0})
        if backbone:
            model = model.base_model
        model.requires_grad_(False)  # 기반 가중치는 고정하고 아래에서 LoRA만 학습 가능하게 한다.
        if checkpoint:
            model = PeftModel.from_pretrained(model, str(checkpoint), is_trainable=trainable)
        elif trainable:
            model = get_peft_model(model, LoraConfig(
                task_type="FEATURE_EXTRACTION" if backbone else "CAUSAL_LM",
                r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules=["q_proj", "v_proj"]))
        if trainable:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            model.print_trainable_parameters()
    if not trainable:
        model.requires_grad_(False)
        model.eval()
    model.config.use_cache = not trainable
    model.config.pad_token_id = tokenizer.pad_token_id
    return model


## 4. 검증과 최종 평가를 분리하는 공통 함수

`PHASE="train"`에서는 별도 검증 200문항으로만 ROUGE-L과 생성 답변을 확인한다.
`PHASE="final"`에서 선택이 완료된 후 공통 최종 50문항에 같은 함수를 사용한다.
`validation_*.csv`와 `test_*.csv`를 구분해 저장한다. 학습 도중 최종 50문항의 점수를 계산하지 않는다.

ROUGE-L F1은 `str.split`과 같은 공백 기준 토큰화로 계산한다.
평가 생성은 `do_sample=False`, `max_new_tokens=EVAL_MAX_NEW_TOKENS`로 통일한다.
질문 이후 새로 생성한 답변만 평가하고, 생성 답변을 사후 정제하지 않는다.
자기소개·회피 문구·반복 비율은 보조 규칙이며 정당한 거절도 포함될 수 있다.
반복은 공백 기준 3-gram이 답변에서 3회 이상 등장한 경우다.


In [ ]:
class WhitespaceTokenizer:
    def tokenize(self, text):
        return text.split()

scorer = rouge_scorer.RougeScorer(["rougeL"], tokenizer=WhitespaceTokenizer())

@torch.inference_mode()
def evaluate_model(model, stage, rows_to_evaluate, split):
    model.eval()
    rows = []
    for item in rows_to_evaluate:
        inputs = tokenizer(make_prompt(item["prompt"]), return_tensors="pt",
                           max_length=MAX_SEQ_LEN, truncation=True).to(device)
        generated = model.generate(**inputs, max_new_tokens=EVAL_MAX_NEW_TOKENS, do_sample=False, num_beams=1,
                                   use_cache=True, pad_token_id=tokenizer.pad_token_id,
                                   eos_token_id=tokenizer.eos_token_id)
        answer = tokenizer.decode(generated[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        grams = list(zip(answer.split(), answer.split()[1:], answer.split()[2:]))
        rows.append({**item, "experiment": EXPERIMENT, "stage": stage, "split": split, "answer": answer,
                     "rougeL": scorer.score(item["reference"], answer)["rougeL"].fmeasure,
                     "self_intro": bool(INTRO.search(answer)),
                     "avoidance": any(word in answer for word in ["답변할 수 없", "알 수 없", "제공할 수 없"]),
                     "repetition": any(grams.count(gram) >= 3 for gram in set(grams))})
    frame = pd.DataFrame(rows)
    frame.to_csv(RESULT_DIR / f"{split}_{EXPERIMENT}_{stage}.csv", index=False, encoding="utf-8-sig")
    display(frame[["rougeL", "self_intro", "avoidance", "repetition"]].mean().to_frame("평균"))
    return frame


In [24]:
# 같은 RUN_NAME의 완료 파일을 확인해, 끝난 학습과 평가를 재사용한다.
def checkpoint_ready(path, reward=False):
    path = Path(path)
    if (path / ".save_in_progress").exists():
        return False
    config = "adapter_config.json" if IS_EXAONE else "config.json"
    weights = ["adapter_model.safetensors", "adapter_model.bin"] if IS_EXAONE else ["model.safetensors", "pytorch_model.bin"]
    ready = (path / config).is_file() and any(
        (path / name).is_file() and (path / name).stat().st_size > 0 for name in weights)
    return ready and (not reward or ((path / "value_head.pt").is_file() and (path / "value_head.pt").stat().st_size > 0))

def result_ready(stage, split, questions):
    path = RESULT_DIR / f"{split}_{EXPERIMENT}_{stage}.csv"
    if not path.exists():
        return False
    try:
        frame = pd.read_csv(path, keep_default_na=False)
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        return False
    columns = {"prompt", "reference", "answer", "experiment", "stage", "split", "rougeL",
               "self_intro", "avoidance", "repetition"}
    if not columns.issubset(frame.columns) or len(frame) != len(questions):
        return False
    if not frame[["prompt", "reference"]].equals(pd.DataFrame(questions)[["prompt", "reference"]]):
        raise ValueError(f"{path.name}: 저장된 평가 문항이 다릅니다. 새 RUN_NAME으로 실행하세요.")
    return (frame.experiment.eq(EXPERIMENT).all() and frame.stage.eq(stage).all()
            and frame.split.eq(split).all() and pd.to_numeric(frame.rougeL, errors="coerce").between(0, 1).all())

def evaluate_saved(checkpoint, stage, questions, split):
    if result_ready(stage, split, questions):
        print(f"{EXPERIMENT} {stage} {split}: 저장된 평가 재사용")
        return
    model = load_model(checkpoint, trainable=False)
    try:
        evaluate_model(model, stage, questions, split)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

# 완전히 저장된 checkpoint만 재개한다. 데이터/설정이 달라지면 RUN_NAME을 바꾼다.
def sft_signature():
    settings = [DATA_SIGNATURE, EXPERIMENT, CFG, MODEL_ID,
                EXAONE_REVISION if IS_EXAONE else KOGPT_REVISION,
                SEED, MAX_SEQ_LEN, SFT_LR, LORA_R, LORA_ALPHA, LORA_DROPOUT]
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()
                          + (DATA_DIR / f"{CFG['data']}.json").read_bytes()).hexdigest()

class CompleteCheckpoint(transformers.TrainerCallback):
    def __init__(self, signature):
        self.signature = signature

    def on_save(self, args, state, control, **kwargs):
        path = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        (path / "save_complete.json").write_text(json.dumps(
            {"step": state.global_step, "signature": self.signature}), encoding="utf-8")
        print(f"Drive checkpoint 저장 완료: {path}")

def latest_sft_checkpoint(directory, signature):
    candidates = sorted((p for p in Path(directory).glob("checkpoint-*")
                         if p.is_dir() and p.name.split("-")[-1].isdigit()),
                        key=lambda p: int(p.name.split("-")[-1]), reverse=True)
    for path in candidates:
        try:
            saved = json.loads((path / "save_complete.json").read_text())
        except (FileNotFoundError, json.JSONDecodeError):
            continue
        if saved.get("signature") != signature:
            raise ValueError(f"{path}: 데이터/설정이 다릅니다. 새 RUN_NAME을 사용하세요.")
        files = ["optimizer.pt", "scheduler.pt", "trainer_state.json", "rng_state.pth"]
        weights = ["adapter_model.safetensors", "adapter_model.bin", "model.safetensors", "pytorch_model.bin"]
        if (saved.get("step") == int(path.name.split("-")[-1])
                and all((path / f).is_file() and (path / f).stat().st_size > 0 for f in files)
                and any((path / f).is_file() and (path / f).stat().st_size > 0 for f in weights)):
            return str(path)
    return None

def save_completed_model(model, path, value_head=None):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    pending = path / ".save_in_progress"
    pending.touch()
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    if value_head is not None:
        torch.save(value_head.state_dict(), path / "value_head.pt")
    pending.unlink()  # 저장 도중 중단되면 checkpoint_ready에서 제외한다.


## 5. SFT: 예제 데이터셋·collator·Trainer 재사용

아래 `SFT_dataset`은 첨부 코드 셀 50에서 prompt 생성 부분만 `make_prompt`로 연결했다.
collator는 코드 셀 51을 그대로 사용한다. 정답 앞의 prompt는 `-100`으로 마스킹한다.
길이 512에서 잘려 정답 토큰이 하나도 남지 않는 샘플은 학습 전에 제외하고 수를 출력한다.


In [ ]:
class SFT_dataset(Dataset):

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        sources = [make_prompt(example[pattern_instruction]) for example in list_data_dict]

        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        examples = [s + t for s, t in zip(sources, targets)]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])


In [ ]:
@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )


In [25]:
def run_sft():
    if PHASE == "train" and DO_TRAIN and checkpoint_ready(SFT_DIR):
        print(f"{EXPERIMENT} SFT: 저장된 모델 재사용")
        evaluate_saved(SFT_DIR, "SFT", validation_questions, "validation")
        return
    if PHASE == "train" and not DO_TRAIN and result_ready("BASE", "validation", validation_questions):
        print(f"{EXPERIMENT} BASE validation: 저장된 평가 재사용")
        return
    if PHASE == "train":
        set_seed(SEED)
        tokenizer.padding_side = "right"
        if DO_TRAIN:
            model = load_model()
            train_dataset = SFT_dataset(str(DATA_DIR / f"{CFG['data']}.json"), tokenizer)
            keep = [i for i, labels in enumerate(train_dataset.labels) if (labels[1:] != -100).any()]
            print("정답이 모두 잘려 제외된 샘플:", len(train_dataset) - len(keep))
            train_dataset.input_ids = [train_dataset.input_ids[i] for i in keep]
            train_dataset.labels = [train_dataset.labels[i] for i in keep]
            data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)
            training_args = transformers.TrainingArguments(
                output_dir=str(MODEL_DIR / "sft_checkpoints"), num_train_epochs=CFG["sft_epochs"],
                per_device_train_batch_size=1 if IS_EXAONE else 8,
                gradient_accumulation_steps=8 if IS_EXAONE else 1,
                per_device_eval_batch_size=8, learning_rate=SFT_LR, warmup_steps=5,
                prediction_loss_only=True, bf16=IS_EXAONE, fp16=not IS_EXAONE, report_to="none", seed=SEED,
                save_strategy="steps", save_steps=SFT_SAVE_STEPS, save_total_limit=2,
                save_only_model=False, remove_unused_columns=False)
            signature = sft_signature()
            resume = latest_sft_checkpoint(training_args.output_dir, signature)
            print("SFT 재개 지점:", resume or "새 학습")
            trainer = transformers.Trainer(model=model, args=training_args,
                data_collator=data_collator, train_dataset=train_dataset,
                callbacks=[CompleteCheckpoint(signature)])
            trainer.train(resume_from_checkpoint=resume)
            save_completed_model(model, SFT_DIR)
            evaluate_model(model, "SFT", validation_questions, "validation")
            del trainer, model, train_dataset, data_collator
        else:
            model = load_model(trainable=False)
            evaluate_model(model, "BASE", validation_questions, "validation")       # A-1: 추가 학습 없이 공통 검증 문항 평가
            del model
        gc.collect()
        torch.cuda.empty_cache()

if not RUN_ALL:
    run_sft()


## 6. RM: 질문 그룹 분할 후 예제 ranking 쌍 생성

예제의 ranking 변환 코드를 그대로 사용한다. 같은 질문의 세 쌍은 모두 미리 정한 질문 분할에 속한다.
**변환된 쌍 전체를 섞어서 train/validation 경계를 나누는 기존 방식은 사용하지 않는다.**
질문 그룹 분할 후 각 분할 안에서 shuffle·수량 제한을 적용한다.

A는 기존 RM 학습 최대 1,000쌍·검증 최대 200쌍을 사용한다.
B/C는 여기에 한국어 UltraFeedback 학습 최대 2,000쌍·검증 최대 200쌍을 추가한다.
한국어 UltraFeedback은 `prompt/chosen/rejected` 필드가 이미 있는 번역·정제 데이터이며, 선호 라벨을 그대로 따른다.
길이 제한 후 chosen/rejected 토큰이 같아지는 쌍은 제외하고 개수를 출력한다.

batch 4, Adam `RM_LR`, 1 epoch와 기존 RewardModelTrainer를 유지한다.
B/C는 같은 초기 모델·데이터·seed로 학습한 RM을 재사용하며 점수층도 저장·복원한다.


In [ ]:
def prepare_reward_data():
    total_data_ranking2chosen = []
    for tmp in rm_rows:
        one_data_ranking2chosen = []

        data = {}
        data['prompt'] = tmp['prompt']
        if tmp['ranking'][0] < tmp['ranking'][1]:
            data['chosen'] = tmp['completion_0']
            data['rejected'] = tmp['completion_1']
        else:
            data['chosen'] = tmp['completion_1']
            data['rejected'] = tmp['completion_0']
        one_data_ranking2chosen.append(data)

        data = {}
        data['prompt'] = tmp['prompt']
        if tmp['ranking'][0] < tmp['ranking'][2]:
            data['chosen'] = tmp['completion_0']
            data['rejected'] = tmp['completion_2']
        else:
            data['chosen'] = tmp['completion_2']
            data['rejected'] = tmp['completion_0']
        one_data_ranking2chosen.append(data)

        data = {}
        data['prompt'] = tmp['prompt']
        if tmp['ranking'][1] < tmp['ranking'][2]:
            data['chosen'] = tmp['completion_1']
            data['rejected'] = tmp['completion_2']
        else:
            data['chosen'] = tmp['completion_2']
            data['rejected'] = tmp['completion_1']
        one_data_ranking2chosen.append(data)



        total_data_ranking2chosen.extend(one_data_ranking2chosen)



    original_pairs = deduplicate(total_data_ranking2chosen, pair=True)
    train_data = take_rows(in_split(original_pairs, "train"), RM_ORIGINAL_TRAIN_PAIRS)
    eval_data = take_rows(in_split(original_pairs, "validation"), RM_VALIDATION_PAIRS)
    if IS_EXAONE:
        train_data = deduplicate(train_data + uf_train, pair=True)
        eval_data = deduplicate(eval_data + uf_validation, pair=True)
    train_keys = {normalize_question(row["prompt"]) for row in train_data}
    validation_keys = {normalize_question(row["prompt"]) for row in eval_data}
    if train_keys & validation_keys:
        raise ValueError("RM의 같은 질문이 train과 validation에 섞였습니다.")
    if len(in_split(train_data, "train")) != len(train_data) or len(in_split(eval_data, "validation")) != len(eval_data):
        raise ValueError("RM 질문 분할이 전체 데이터 분할과 다릅니다.")
    print("RM 학습/검증 쌍:", len(train_data), len(eval_data), "/ 질문 그룹 중복:", len(train_keys & validation_keys))
    train_data = [{**row, "prompt": make_prompt(row["prompt"])} for row in train_data]
    eval_data = [{**row, "prompt": make_prompt(row["prompt"])} for row in eval_data]
    return train_data, eval_data

if not RUN_ALL:
    train_data, eval_data = prepare_reward_data()


In [ ]:
# RewardModel/Critic의 평균 점수 계산을 재사용하되 padding을 평균에서 제외한다.
def sequence_value(module, sequences, attention_mask=None):
    if attention_mask is None:
        attention_mask = sequences.ne(tokenizer.pad_token_id).long()
    position_ids = (attention_mask.cumsum(-1) - 1).clamp_min(0)
    hidden = module.model(input_ids=sequences, attention_mask=attention_mask,
                          position_ids=position_ids, use_cache=False)["last_hidden_state"]
    values = module.value_head(hidden.float()).squeeze(-1)
    return (values * attention_mask).sum(-1) / attention_mask.sum(-1).clamp_min(1)

class ProjectRewardModel(RewardModel):
    def forward(self, sequences, attention_mask=None):
        return sequence_value(self, sequences, attention_mask)

class ProjectCritic(Critic):
    def forward(self, sequences, action_mask=None, attention_mask=None):
        return sequence_value(self, sequences, attention_mask)

def load_reward(checkpoint=None, critic=False, trainable=True):
    backbone = load_model(checkpoint, backbone=True, trainable=trainable)
    head = nn.Linear(backbone.config.hidden_size, 1).to(device)
    if checkpoint:
        head.load_state_dict(torch.load(Path(checkpoint) / "value_head.pt",
                                        map_location=device, weights_only=True))
    cls = ProjectCritic if critic else ProjectRewardModel
    model = cls(backbone, head)
    if not trainable:
        model.requires_grad_(False)
        model.eval()
    return model


In [26]:
def run_rm():
    if PHASE == "train" and DO_TRAIN:
        if not checkpoint_ready(RM_DIR, reward=True):
            set_seed(SEED)
            tokenizer.padding_side = "right"
            model = load_reward()
            train_dataset = RewardDataset(train_data, tokenizer, MAX_SEQ_LEN)
            eval_dataset = RewardDataset(eval_data, tokenizer, MAX_SEQ_LEN)
            for name, dataset in [("학습", train_dataset), ("검증", eval_dataset)]:
                keep = [i for i in range(len(dataset)) if not torch.equal(
                    dataset.chosen[i]["input_ids"], dataset.reject[i]["input_ids"])]
                print(f"RM {name}: 길이 제한 후 동일해진 쌍 {len(dataset) - len(keep)}개 제외")
                dataset.chosen = [dataset.chosen[i] for i in keep]
                dataset.reject = [dataset.reject[i] for i in keep]
                if not keep:
                    raise ValueError(f"RM {name} 쌍이 남지 않았습니다. MAX_SEQ_LEN과 원자료를 확인하세요.")
            trainer = RewardModelTrainer(model=model, strategy=NaiveStrategy(),
                optim=torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=RM_LR),
                train_dataset=train_dataset, eval_dataset=eval_dataset, batch_size=4, max_epochs=1)
            trainer.fit(use_lora=0)    # 이 인수는 예제 loralib용. EXAONE PEFT 어댑터는 이미 적용됨.
            save_completed_model(model.model, RM_DIR, model.value_head)
            del trainer, model, train_dataset, eval_dataset
        print("PPO에 사용할 학습된 RM:", RM_DIR)
    gc.collect()
    torch.cuda.empty_cache()

if not RUN_ALL:
    run_rm()


## 7. PPO: 선택한 KoAlpaca 학습 질문을 재사용

A는 기존 PPO의 학습 그룹 질문을 사용한다. B/C는 여기에 **SFT에 추가한 KoAlpaca 질문을 그대로** 합치고 중복을 제거한다.
검증·최종 평가 질문은 PPO 질문 풀에도 들어가지 않는다.

예제의 `max_epochs=1`, `num_episodes=10`, `max_timesteps=3`, `update_timesteps=3`, 두 batch 8을 유지했다.
현재 설정의 총 질문 추출은 **10 × 3 × 8 = 240회**이며, 회차 사이에는 같은 질문이 다시 뽑힐 수 있다.
추가한 5,000개 질문을 전부 사용한다는 뜻으로 해석하지 않는다. 실제 학습량은 질문 풀 크기와 함께 출력한다.
반복 수와 입력·답변 길이는 이번 하이퍼파라미터 논의에서 우선 결정할 항목이다.

PPO 입력 최대 96·전체 128 토큰은 예제 시작값이다. 입력이 96이면 생성 여유는 최대 32토큰이다.
긴 KoAlpaca 질문과 EXAONE 대화 템플릿에 충분한지 확인한 뒤, B/C에 같은 길이와 반복 수를 적용한다.
네 모델을 함께 올리는 실제 A100 40GB 메모리 사용량은 아직 측정하지 않았다.
현재 batch·길이에서 먼저 실행하고 최대 메모리를 확인한 뒤 후속 실험의 길이를 조정한다.

생성 결과가 일찍 끝나면 수집 회차마다 길이가 달라진다.
예제 버퍼에서 `sequences`와 `attention_mask`도 기존 `zero_pad_sequences`로 왼쪽 패딩하도록 보완했다.
추가된 위치는 attention/action mask의 0으로 계산에서 제외한다. 기존 확률·답변·마스크의 오른쪽 정렬을 유지한다.


In [ ]:
class ProjectActor(Actor):
    @torch.no_grad()
    def generate(self, input_ids, return_action_mask=True, **kwargs):
        kwargs.pop("prepare_inputs_fn", None)
        kwargs.pop("update_model_kwargs_fn", None)
        sequences = self.model.generate(input_ids=input_ids, **kwargs)
        attention_mask = sequences.ne(kwargs["pad_token_id"]).long()
        if not return_action_mask:
            return sequences, attention_mask, None
        input_len = input_ids.size(1)
        # 예제의 mask와 같이 첫 EOS까지 포함하고 그 뒤는 제외한다.
        eos = sequences[:, input_len:].eq(kwargs["eos_token_id"])
        action_mask = (eos.cumsum(-1) - eos.long()).eq(0)
        return sequences, attention_mask, action_mask

    def forward(self, sequences, num_actions, attention_mask=None):
        position_ids = (attention_mask.cumsum(-1) - 1).clamp_min(0)
        output = self.model(input_ids=sequences, attention_mask=attention_mask,
                            position_ids=position_ids, use_cache=False)
        log_probs = log_probs_from_logits(output["logits"][:, :-1, :].float(), sequences[:, 1:])
        return log_probs[:, -num_actions:]

def tokenize_fn(texts):
    tokenizer.padding_side = "left"
    batch = tokenizer(texts, return_tensors="pt", max_length=PPO_PROMPT_LEN, padding=True, truncation=True)
    return {key: value.to(device) for key, value in batch.items()}


In [27]:
def run_ppo():
    if PHASE == "train" and DO_TRAIN and checkpoint_ready(PPO_DIR):
        print(f"{EXPERIMENT} PPO: 저장된 모델 재사용")
        evaluate_saved(PPO_DIR, "PPO", validation_questions, "validation")
        return
    if PHASE == "train" and DO_TRAIN:
        set_seed(SEED)
        actor = ProjectActor(load_model(SFT_DIR))
        initial_model = ProjectActor(load_model(SFT_DIR, trainable=False))
        critic = load_reward(RM_DIR, critic=True)
        reward_model = load_reward(RM_DIR, trainable=False)
        actor_optim = torch.optim.Adam((p for p in actor.parameters() if p.requires_grad), lr=PPO_ACTOR_LR)
        critic_optim = torch.optim.Adam((p for p in critic.parameters() if p.requires_grad), lr=PPO_CRITIC_LR)
        list_prompt = [make_prompt(row["prompt"]) for row in ppo_rows]

        trainer = PPOTrainer(NaiveStrategy(), actor, critic, reward_model, initial_model,
            actor_optim, critic_optim, max_epochs=PPO_EPOCHS,
            train_batch_size=PPO_BATCH_SIZE, experience_batch_size=PPO_EXPERIENCE_BATCH_SIZE,
            kl_coef=PPO_KL_COEF, eps_clip=PPO_EPS_CLIP, value_clip=PPO_VALUE_CLIP,
            tokenizer=tokenize_fn, max_length=PPO_TOTAL_LEN, do_sample=True, temperature=1.0, top_k=50,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, use_cache=True)
        print("PPO 질문 풀:", len(list_prompt), "/ 총 추출 횟수:",
              PPO_EPISODES * PPO_TIMESTEPS * PPO_EXPERIENCE_BATCH_SIZE)
        trainer.fit(list_prompt, num_episodes=PPO_EPISODES,
                    max_timesteps=PPO_TIMESTEPS, update_timesteps=PPO_UPDATE_TIMESTEPS)
        save_completed_model(actor.model, PPO_DIR)

        # 평가 전에 고정 모델·critic·optimizer 참조를 해제한다.
        del trainer, critic, reward_model, initial_model, actor_optim, critic_optim
        gc.collect()
        torch.cuda.empty_cache()
        evaluate_model(actor.model, "PPO", validation_questions, "validation")
        del actor
    gc.collect()
    torch.cuda.empty_cache()

if not RUN_ALL:
    run_ppo()


## 8. 검증으로 에포크 선택 → 최종 50문항 평가

학습 단계에서 B/C의 **SFT 직후 공통 검증 ROUGE-L**을 비교해 epoch를 선택한다.
PPO 이후 검증 결과도 저장하지만 이 답안의 epoch 선택 기준은 SFT 검증 점수로 고정했다.
자동 실행에서는 선택 파일을 만든 뒤 `PHASE="final"`로 전환해 네 비교군을 평가한다. 수동 실행은 `RUN_ALL=False`에서 진행한다.
최종 결과를 보고 설정을 다시 맞추면 그 50문항은 이후 검증용으로 취급하고 새 최종 평가셋을 확보한다.

- A ↔ B: 모델·SFT/RM/PPO 데이터 구성을 함께 바꾼 전체 개선안 비교.
- A-1 ↔ B/C: 배포 EXAONE 4.0 1.2B에 프로젝트 학습을 추가한 효과.
- B ↔ C: 같은 데이터·RM·PPO 설정에서 SFT 1 epoch ↔ 3 epochs 비교.
- 각 실험의 SFT ↔ PPO: PPO 적용 전후 답변·점수 변화.


In [ ]:
selection_path = RESULT_DIR / "epoch_selection.json"

def select_epoch():
    if PHASE == "train":
        candidates = {}
        for name in ["B", "C"]:
            path = RESULT_DIR / f"validation_{name}_SFT.csv"
            if path.exists():
                frame = pd.read_csv(path, keep_default_na=False)
                if not frame[["prompt", "reference"]].equals(pd.DataFrame(validation_questions)):
                    raise ValueError("B/C 검증 문항이 다릅니다. 같은 검증 파일로 다시 평가하세요.")
                candidates[name] = frame.rougeL.mean()
        if len(candidates) == 2:
            selected = max(["B", "C"], key=lambda name: candidates[name])
            save_json(selection_path, {"criterion": "SFT validation ROUGE-L",
                "validation_scores": candidates, "selected_experiment": selected,
                "selected_sft_epochs": EXPERIMENTS[selected]["sft_epochs"], "data_signature": DATA_SIGNATURE})
            print("검증으로 선택한 SFT epoch:", EXPERIMENTS[selected]["sft_epochs"], candidates)
        else:
            print("B와 C의 SFT 검증 결과가 모이면 epoch 선택을 저장합니다.")

if not RUN_ALL:
    select_epoch()


In [ ]:
def run_final_evaluation():
    if PHASE == "final":
        if not selection_path.exists():
            raise RuntimeError("먼저 train 단계에서 B/C 검증과 epoch 선택을 완료하세요.")
        selection = read_json(selection_path)
        if selection["data_signature"] != DATA_SIGNATURE:
            raise ValueError("선택 당시 데이터와 다릅니다. 검증·선택부터 다시 진행하세요.")
        print("최종 평가 전에 고정한 선택:", selection["selected_experiment"], selection["selected_sft_epochs"])
        stages = [("SFT", SFT_DIR), ("PPO", PPO_DIR)] if DO_TRAIN else [("BASE", None)]
        for stage, checkpoint in stages:
            if checkpoint is not None and not checkpoint_ready(checkpoint):
                raise FileNotFoundError(f"학습 체크포인트가 없습니다: {checkpoint}")
            evaluate_saved(checkpoint, stage, evaluation, "test")

if not RUN_ALL:
    run_final_evaluation()


In [ ]:
def show_comparison():
    result_split = "validation" if PHASE == "train" else "test"
    questions = validation_questions if PHASE == "train" else evaluation
    frames, summary = [], []
    for experiment, stages in {"A": ["SFT", "PPO"], "A-1": ["BASE"],
                               "B": ["SFT", "PPO"], "C": ["SFT", "PPO"]}.items():
        for stage in stages:
            path = RESULT_DIR / f"{result_split}_{experiment}_{stage}.csv"
            row = {"experiment": experiment, "stage": stage, "status": "미실행"}
            if path.exists():
                frame = pd.read_csv(path, keep_default_na=False)
                expected = pd.DataFrame(questions)[["prompt", "reference"]]
                if not frame[["prompt", "reference"]].equals(expected):
                    raise ValueError(f"{path.name}: 평가 질문/참고 답변이 다릅니다. 같은 평가 파일로 다시 평가하세요.")
                frames.append(frame)
                row.update(status="완료", n=len(frame), rougeL=frame.rougeL.mean(),
                           self_intro_pct=100 * frame.self_intro.mean(),
                           avoidance_pct=100 * frame.avoidance.mean(), repetition_pct=100 * frame.repetition.mean())
            summary.append(row)
    summary = pd.DataFrame(summary)
    display(summary)
    summary.to_csv(RESULT_DIR / f"{result_split}_summary.csv", index=False, encoding="utf-8-sig")
    if frames:
        answers = pd.concat(frames, ignore_index=True)
        answers["model_stage"] = answers.experiment + "_" + answers.stage
        wide = answers.pivot(index="prompt", columns="model_stage", values="answer")
        comparison = pd.DataFrame(questions).join(wide, on="prompt")
        comparison.to_csv(RESULT_DIR / f"{result_split}_answers_all.csv", index=False, encoding="utf-8-sig")
        final_columns = [name for name in ["A_PPO", "A-1_BASE", "B_PPO", "C_PPO"] if name in comparison]
        with pd.option_context("display.max_colwidth", None):
            display(comparison[["prompt", "reference"] + final_columns].head(5))
    return summary

if not RUN_ALL:
    show_comparison()


## 9. 전체 자동 실행

`RUN_ALL=True`이면 앞의 학습 셀들은 함수를 정의하고, 이 셀이 A → A-1 → B → C의 학습·검증 → 에포크 선택 → 최종 평가를 순서대로 실행한다.
같은 `RUN_NAME`의 모델과 평가 CSV가 준비된 단계는 재사용한다. 모델만 저장되고 평가가 중단됐다면 평가부터 진행한다.
오류·사용자 중단 시 멈추며, 재실행하면 저장되지 않은 학습 단계는 처음부터 다시 시작한다. 학습 중간 optimizer 상태까지 이어받는 기능은 포함하지 않는다.
설치 후 재시작·Drive 접근 허용을 마치면 이후 코드 셀들을 한 번에 실행할 수 있다.
이미 학습한 Colab 런타임에서는 함께 제공한 `LLM_Trend_전체실행.py`를 `/content`에 업로드한 뒤 새 셀에서 아래 한 줄을 실행해도 된다.

```python
%run -i /content/LLM_Trend_전체실행.py
```

같은 RUN_NAME과 작업 경로를 유지해야 기존 A 모델·평가를 재사용한다. 모델·결과는 Colab 작업 폴더에 있으므로 런타임 삭제 전에 따로 보관한다.


In [29]:
def run_all_experiments():
    global PHASE, tokenizer, train_data, eval_data, ppo_rows
    PHASE = "train"
    for name in ["A", "A-1", "B", "C"]:
        select_experiment(name)
        print(f"\n===== {name}: 학습·검증 =====")
        tokenizer = prepare_tokenizer()
        ppo_rows = ppo_improved if IS_EXAONE else ppo_original
        if DO_TRAIN:
            train_data, eval_data = prepare_reward_data()
        try:
            run_sft()
            run_rm()
            run_ppo()
        except BaseException as error:
            # 오류 화면이 이전 모델을 계속 붙잡지 않도록 종료된 함수의 참조를 해제한다.
            traceback.clear_frames(error.__traceback__)
            raise
        finally:
            gc.collect()
            torch.cuda.empty_cache()

    # B/C 검증이 모두 끝난 뒤 선택하고, 이후에만 최종 50문항을 평가한다.
    select_epoch()
    show_comparison()
    PHASE = "final"
    for name in ["A", "A-1", "B", "C"]:
        select_experiment(name)
        print(f"\n===== {name}: 최종 평가 =====")
        tokenizer = prepare_tokenizer()
        try:
            run_final_evaluation()
        except BaseException as error:
            # 오류 화면이 이전 모델을 계속 붙잡지 않도록 종료된 함수의 참조를 해제한다.
            traceback.clear_frames(error.__traceback__)
            raise
        finally:
            gc.collect()
            torch.cuda.empty_cache()
    print("전체 실험 완료. 최종 결과:", RESULT_DIR)
    return show_comparison()

if RUN_ALL:
    final_summary = run_all_experiments()



===== A: 학습·검증 =====


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


한국어 복원: 안녕하세요. 한국어 문장을 확인합니다.
RM 학습/검증 쌍: 1000 200 / 질문 그룹 중복: 0
A SFT: 저장된 모델 재사용
A SFT validation: 저장된 평가 재사용
PPO에 사용할 학습된 RM: /content/drive/MyDrive/llm_trend_project/comparison_02_group_split/a100_exaone4_1_2b_lora/models/rm_kogpt2
A PPO: 저장된 모델 재사용
A PPO validation: 저장된 평가 재사용

===== A-1: 학습·검증 =====
한국어 복원: 안녕하세요. 한국어 문장을 확인합니다.
A-1 BASE validation: 저장된 평가 재사용

===== B: 학습·검증 =====
한국어 복원: 안녕하세요. 한국어 문장을 확인합니다.
RM 학습/검증 쌍: 3000 400 / 질문 그룹 중복: 0
B SFT: 저장된 모델 재사용
B SFT validation: 저장된 평가 재사용
PPO에 사용할 학습된 RM: /content/drive/MyDrive/llm_trend_project/comparison_02_group_split/a100_exaone4_1_2b_lora/models/rm_exaone_shared
B PPO: 저장된 모델 재사용
B PPO validation: 저장된 평가 재사용

===== C: 학습·검증 =====
한국어 복원: 안녕하세요. 한국어 문장을 확인합니다.
RM 학습/검증 쌍: 3000 400 / 질문 그룹 중복: 0


trainable params: 1,597,440 || all params: 1,280,988,928 || trainable%: 0.1247


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


...

,평균
rougeL,0.078728
self_intro,0.000000
avoidance,0.000000
repetition,0.620000


,평균
rougeL,0.080038
self_intro,0.000000
avoidance,0.000000
repetition,0.580000


전체 실험 완료. 최종 결과: /content/drive/MyDrive/llm_trend_project/comparison_02_group_split/a100_exaone4_1_2b_lora/results


,experiment,stage,status,n,rougeL,self_intro_pct,avoidance_pct,repetition_pct
0,A,SFT,완료,50,0.055633,26.0,8.0,42.0
1,A,PPO,완료,50,0.046964,34.0,16.0,38.0
2,A-1,BASE,완료,50,0.055935,0.0,0.0,0.0
3,B,SFT,완료,50,0.072790,0.0,0.0,58.0
4,B,PPO,완료,50,0.073582,0.0,0.0,56.0
5,C,SFT,완료,50,0.078728,0.0,0.0,62.0
6,C,PPO,완료,50,0.080038,0.0,0.0,58.0


,prompt,reference,A_PPO,A-1_BASE,B_PPO,C_PPO
0,바닷가에서 본 납작한 모양의 뼈는 화석일까요? 해당 모양의 명칭이 무엇인지 알려주세요.,"그것은 화석이 아니라 연잎성게류의 골격(뼈)입니다. 연잎성게류는 바닷가에서 주로 발견되며, 모래 속을 파고들어 입까지 가는 유기물을 먹습니다. 이들의 주요 특징은 화문(花紋)이라고 하는 5개의 구멍으로 배치된 부분이 방사상으로 배열되어 있는 것입니다. 이러한 종류의 성게류는 sand dollar라는 이름으로도 불리며, 모래 기질에 굴을 파는 데 잘 적응하고 있습니다.",'바닷가에서 본 납작한 모양의 뼈는 화석일 가능성이 높습니다. 화석일 가능성이 높습니다.,"바닷가에서 관찰된 납작한 모양의 뼈는 일반적으로 **화석**으로 분류됩니다. 특히 이 모양은 다음과 같은 화석 유형과 연관될 수 있습니다:\n\n1. **화석화된 두개골 화석**: 특히 고대 해양 생물의 두개골에서 흔히 발견되는 납작한 부분으로, 예를 들어 공룡이나 초기 포유류의 뇌 구조를 보존한 경우가 많습니다.\n\n2. **이완골 화석**: 뼈의 끝 부분이 평평하게 변형된 형태로, 일부 해양 무척추동물이나 어류에서 관찰됩니다.\n\n3.","해당 모양의 명칭은 '화석'입니다. 화석은 과거에 존재했던 생물의 뼈, 껍질, 껍질 조각 등을 보존한 것입니다. 이 화석은 바다에서 발견되는 경우가 많으며, 바다 생물의 뼈나 껍질을 보존한 것입니다.","해당 뼈는 화석입니다. 이 뼈는 화석으로, 화석은 과거에 존재했던 생물의 뼈나 화석으로 된 화석입니다. 이 뼈는 납작한 모양으로, 화석으로 분류됩니다. 이 뼈는 바다에서 발견되었으며, 화석으로 분류되는 것은 과거에 존재했던 생물의 뼈나 화석입니다. 이 뼈는 바다에서 발견되었으며, 화석으로 분류됩니다. 이 뼈는 과거에 존재했던 생물의 뼈나 화석입니다. 이 뼈는 바다에서 발견되었으며, 화석으로 분류됩니다. 이 뼈는 과거에 존재했던 생물"
1,"근육이 커지는 원리는 무엇인가요? 근육이 찢어지면 회복 과정에서 어떻게 강해지는 건가요?\n제가 대충 알기로는 강한 힘을 주면 근육섬유가 찢어지고 회복되면서 강해진다고 하는데, 이 과정이 어떤 방식으로 이루어지는지 궁금합니다.","근육이 커지는 원리는 근육섬유가 찢어지고 회복되면서 부풀어 오르는 것입니다. 근육에는 연성 근육과 강성 근육이 있으며, 강성 근육은 부상한 부위에 살이 차면서 형성되는 근육으로 순간적인 힘만을 발휘할 수 있습니다. 반면에, 연성 근육은 꾸준히 힘을 발휘할 수 있으며, 비가 오면 강성 근육은 아픔을 느끼지만, 연성 근육은 더욱 상쾌하고 부드러워집니다. 따라서, 몸을 만들기 위해 운동을 하는 것은 건강을 지키는 좋은 방법이 될 수 있다고 할 수 있습니다.",'근육이 찢어지는 원리는 다음과 같습니다.\n\n1. 근육을 수축시키기: 근육을 수축시키기 위해서는 근육을 수축시키기 위해 근육을 수축시켜야 합니다. 근육을 수축시키기 위해서는 근육을 수축시키기 위해 근육을 수축시켜야 합니다.\n\n2. 근육을 수축시키기: 근육을 수축시키기 위해서는 근육을 수축시키기 위해 근육을 수축시켜야 합니다. 근육을 수축시키기 위해서는 근육을 수축시키기 위해 근육을 수축시켜야 합니다.\n\n3. 근육을 수축시키기: 근육을 수축시키기 위해서는 근육을 수축시키기 위해서는 근육을 수축시켜야 합니다. 근육을 수축시키기 위해서는 근육을 수축시키기 위해서는 근육을 수축시키기,"근육이 커지고 강해지는 과정은 주로 **단백질 합성 증가**와 **세포 내 대사 변화**를 통해 이루어집니다. 특히 운동 후 회복 과정에서는 다음과 같은 생리학적 메커니즘이 작용합니다:\n\n### 1. **근육 성장의 기본 원리**\n - **미오섬유 수축**: 근섬유(주로 근형성 섬유)가 지속적인 수축을 반복하면 미세한 수축 단위(미크로파괴)가 발생합니다. 이 과정에서 세포 내 칼슘 이온 농도가 증가하며, 미토콘드리아가 활성화되어 ATP 생산이 촉진됩니다.","근육이 커지는 원리는 근육섬유가 손상되면, 손상된 근육섬유는 회복되면서 새로운 근육섬유가 성장하는 것입니다. 이 과정은 다음과 같은 단계를 거칩니다.\n\n1. 손상된 근육섬유는 손상되면서 세포 내에서 염증 반응이 일어나고, 이로 인해 세포가 분열하고 새로운 세포가 생성됩니다.\n2. 새로운 세포는 손상된 근육섬유를 대체하게 되며, 이로 인해 근육이 더 강해집니다.\n3. 이 과정은 근육이 더 강해지는 것이 아니라, 손상된 근육섬유를","근육이 커지는 원리는 근육섬유가 손상되면, 손상된 근육섬유를 대체하는 새로운 근육섬유가 생성되어 근육이 커지는 것입니다. 이 과정은 근육이 손상되면, 손상된 근육섬유를 대체하는 새로운 근육섬유가 생성되어 근육이 커지는 것입니다. 이 과정은 근육이 손상되면, 손상된 근육섬유를 대체하는 새로운 근육섬유가 생성되어 근육이 커지는 것입니다. 이 과정은 근육이 손상되면, 손상된 근육섬유를 대체하는 새로운 근육섬유가 생성되어 근육이 커지는 것입니다."
2,"미용실, 미장원, 미용원, 이발소, 이발관 등 이름이 다양한 이유는 무엇인가요?","미용실, 미장원, 미용원, 이발소, 이발관 등은 모두 머리를 손질하는 곳으로 이름이 다양한 이유는 쓰는 사람의 소지한 자격과 성향에 따라 다르기 때문입니다. 업무 범위를 기준으로 분류하면 미용 업무를 하는 곳이 미용실, 미장원, 미용원이고, 이용 업무를 하는 곳이 이발소, 이발관, 이용원입니다.\n\n미용 업무는 미용사 자격증을 취득한 사람이 행하며, 머리를 자르고, 세련되게 모양을 내고, 염색하는 등 미용과 관련된 일을 합니다. 반면, 이용 업무는 미용사 자격증을 취득한 사람이 이발, 식도락 등과 같은 일을 합니다.\n\n따라서, 이름이 다양한 이유는 쓰는 사람의 성향과 업무 범위에 따라 다르다는 것입니다.","'미용실, 미장원, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이발소, 이","미용실, 미장원, 미용원, 이발소, 이발관 등 이름이 다양한 이유는 다음과 같은 여러 가지 요인이 복합적으로 작용한 결과입니다:\n\n1. **서비스 범위의 다양성**: 각 시설은 주로 특정한 서비스를 전문적으로 제공합니다. 예를 들어, 미용실은 haircuts, 염색, 스타일링 등을, 이발소는 면도, 머리 자르기 등을 주로 합니다. 이러한 특화된 기능을 반영한 이름이 붙었습니다.\n\n2. **지역적 특성 반영**: 이름은 해당 지역의 문화, 역사, 또는 지역적인 특성을","미용, 미용, 이발, 미용원, 미용원, 미용원, 이발소, 이발관 등 이름이 다양한 이유는 무엇일까요? \n\n1. 업종의 다양성: 미용, 이발, 미용원, 미용원, 이발소, 이발관 등은 모두 미용과 관련된 업종입니다. 하지만 미용, 이발, 미용원, 미용원, 이발소, 이발관 등은 각각 다른 업종을 가지고 있습니다. 미용은 미용, 미용원, 미용관, 미용원, 미용관, 미용원, 미용관, 미용관, 미용관, 미용관, 미용관, 미용","미용업은 다양한 서비스를 제공하는 업종으로, 미용실, 미장원, 미용원, 이발소, 이발관 등 이름이 다양하게 사용되고 있습니다. 이는 각 업종의 특성과 서비스의 종류에 따라 다르게 명칭이 부여되기 때문입니다. 예를 들어, 미용실은 주로 머리를 다듬고 스타일링하는 서비스를 제공하는 곳이고, 미장원은 피부를 건강하게 관리하는 서비스를 제공하는 곳입니다. 이러한 다양한 서비스를 제공하는 다양한 미용실, 미장원, 미용원, 이발소, 이발관 등이"
3,"전통적인 방법으로 만든 된장의 숙성과 유통기간의 차이는 무엇인가요? 뉴스에서 보면 유통기한이 지난 된장을 유통시킨 업체가 있지만, 이는 첨가물의 유무에 따라 어떻게 달라지는 건가요?\n가끔 TV에서 30-40년 된 된장을 볼 수 있는데, 그 안에 푸른 곰팡이가 생겨도 먹을 수 있을까요?","전통식 된장은 메주를 띄워 소금물에 두고 

### 답변 5개를 보고 직접 작성할 해석

1. 질문에 맞는 사실을 답했는가? 참고 답변 자체에 오류가 있는가?
2. 도입문·불필요한 회피·반복이 실제로 줄었는가?
3. A-1과 비교해 B/C의 답변이 좋아졌는가? 낮아진 항목은 무엇인가?
4. SFT 직후보다 PPO 이후 좋아진 답변과 나빠진 답변은 무엇인가?
5. B보다 C가 좋아졌는가? ROUGE-L 차이와 실제 답변이 같은 방향인가?

한 seed의 소규모 비교로 기록한다. epoch 선택은 검증 점수로 완료하고 최종 50문항은 마지막 비교에 쓴다.

### 다음에 논의할 하이퍼파라미터

| 항목 | 현재 시작값 | 우선 확인할 점 |
|---|---|---|
| SFT/RM 최대 길이 | 512 | 질문·답변이 얼마나 잘리는지, RM의 두 답변이 같아지는 비율 |
| PPO 입력/전체 길이 | 96/128 | EXAONE 템플릿과 긴 질문, 충분한 답변 길이 |
| PPO 경험 수 | 10 × 3 × 8 = 240 | 추가 질문 5,000개 중 실제 노출되는 양 |
| SFT / RM 학습률 | 각 5e-5 | 손실 안정성과 검증 성능; 바꾸면 한 단계씩 비교 |
| PPO actor / critic 학습률 | 각 5e-6 | actor 변화와 critic 추정 안정성 |
| SFT 유효 batch | 8; EXAONE 실제 1 × 누적 8 | 메모리를 조정할 때 유효 batch 유지 |
| LoRA r / alpha / dropout | 8 / 16 / 0.05 | 우선 고정하고 데이터·길이·학습률부터 확인 |
| PPO KL / clipping | 0.1 / policy 0.2, value 0.4 | SFT 모델에서 과하게 벗어나는지; 현재 예제는 고정 KL 계수 |
| warmup | 5 steps | 데이터 증가 후 전체 update 대비 비율 확인 |
| 평가 생성 | greedy, 새 토큰 128 | 모든 모델에 같은 조건 유지 |

위 값은 이번 수정에서 유지한 예제 기반 시작값이다. 후보 값은 검증 결과와 GPU 메모리를 보고 논의한다.

### 저장되는 파일

`data/`는 Google Drive의 `내 드라이브/llm_trend_project/<RUN_NAME>/data/`에 있다. `models/`, `results/`는 같은 Drive의 `<RUN_NAME>/a100_exaone4_1_2b_lora/`에 저장한다.

- SFT는 200 optimizer step마다 모델·optimizer·scheduler·난수 상태를 저장하고 최근 2개를 보관한다.
- 세션 종료 후 같은 `RUN_NAME`으로 위에서부터 실행하면 완료된 단계는 재사용하고, 진행 중이던 SFT는 마지막으로 저장 완료된 checkpoint부터 재개한다.
- RM/PPO는 단계 완료 시 저장한다. 해당 단계 도중 세션이 종료되면 그 단계의 학습은 처음부터 다시 실행한다.
- 새 데이터·하이퍼파라미터 실험에서는 `RUN_NAME`을 바꾼다.

`models/`, `results/`는 기존 Colab 작업 폴더(`llm_trend_project/<RUN_NAME>/a100_exaone4_1_2b_lora/`)에 있다.

- `data/kochatgpt/`: 예제 SFT·RM·PPO 원자료 3개.
- `data/koalpaca.json`, `data/ultrafeedback.json`: 고정 버전 추가 데이터의 사용 필드.
- `data/hf_datasets_cache/`: Hugging Face datasets 캐시.
- `data/question_splits.json`: 전체 출처에 공통 적용한 질문 분할.
- `data/validation_questions.json`: 별도 검증 200문항.
- `data/evaluation_50.json`: 마지막에 사용할 최종 50문항.
- `data/koalpaca_train.json`, `ultrafeedback_train.json`: 실제 선택한 추가 학습 데이터.
- `models/A|B|C/sft`, `models/A|B|C/ppo`: KoGPT-2는 모델, EXAONE은 LoRA 어댑터와 토크나이저.
- `models/rm_kogpt2`, `models/rm_exaone_shared`: RM 본체/어댑터와 `value_head.pt`.
- `results/validation_*.csv`, `test_*.csv`: 검증/최종 평가를 구분한 문항별 답변·점수와 비교표.
- `results/epoch_selection.json`: 검증으로 선택한 epoch와 기준 점수.

### 공식 구현 참고

- [KoChatGPT 예제](https://github.com/airobotlab/KoChatGPT/tree/5d01e3d74d5ef5a0a32c18150dc9b907eef3f691/colossalai_ChatGPT_230319)
- [EXAONE 모델 및 고정 버전](https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B/tree/3abf2810673c7c0778df64a73c2d52eab32d91c4)
- [한국어 UltraFeedback 데이터](https://huggingface.co/datasets/maywell/ko_Ultrafeedback_binarized)
- [KoAlpaca 데이터](https://huggingface.co/datasets/beomi/KoAlpaca-v1.1a)
- [PEFT LoRA](https://huggingface.co/docs/peft/v0.15.0/en/developer_guides/lora)
- [PEFT 가중치 정밀도](https://huggingface.co/docs/peft/v0.15.0/en/developer_guides/troubleshooting#dtype-related-issues)
- [ROUGE의 공백 토크나이저 연결 구현](https://huggingface.co/spaces/evaluate-metric/rouge/blob/main/rouge.py)
